# 📰 Google News Scraper - LPDP Articles

---

## 🎯 Tujuan Project
Mengumpulkan artikel berita tentang **LPDP (Lembaga Pengelola Dana Pendidikan)** dari Google News untuk keperluan analisis data kelompok.

### Target (dengan 20 Keywords):
- **~5000 artikel mentah** (~250 artikel × 20 kata kunci)
- Setelah deduplikasi: **~2500 artikel unik** tentang LPDP
- Distribusi ke **5 anggota kelompok** (masing-masing ~500 artikel)
- Format output: **CSV file** dengan struktur data yang bersih dan terurut

> ⚠️ **Note**: Google News API membatasi hasil query ke **~100-250 artikel max** per keyword, terlepas dari setting `max_results`. Strategi yang digunakan: **20 variasi keyword** untuk memaksimalkan coverage artikel.

---

## 📦 Import Libraries

Library yang digunakan:
- **GNews**: Untuk scraping artikel dari Google News
- **Pandas**: Untuk manipulasi data
- **NumPy**: Untuk operasi numerik
- **Time**: Untuk delay antar request

In [1]:
from gnews import GNews
import pandas as pd
import numpy as np
import time

print("✅ Libraries berhasil diimport!")

✅ Libraries berhasil diimport!


## 🔧 Konfigurasi Scraper

Mengatur parameter scraper:
- **Language**: Indonesia (`id`)
- **Country**: Indonesia (`ID`)
- **Max Results**: 500 (setting library)

> ⚠️ **Limitasi Google News API**: Meskipun `max_results=500`, Google News RSS feed **hanya mengembalikan ~100-250 artikel** per query. Ini adalah batasan dari Google News itu sendiri, bukan dari code.

In [20]:
print("⚙️ Menginisiasi Google News Scraper...")

# Konfigurasi GNews untuk region Indonesia
google_news = GNews(language='id', country='ID', max_results=500)

print("✅ Scraper siap digunakan!")

⚙️ Menginisiasi Google News Scraper...
✅ Scraper siap digunakan!


## 🔍 Kata Kunci Pencarian

Menggunakan **20 variasi kata kunci** untuk memaksimalkan coverage artikel LPDP:

### 20 keywords LPDP
| No | Kata Kunci | Deskripsi |
|----|------------|-----------|
| 1 | LPDP | Kata kunci umum |
| 2 | Beasiswa LPDP | Fokus program beasiswa |
| 3 | Awardee LPDP | Fokus penerima beasiswa |
| 4 | Wawancara LPDP | Proses seleksi |
| 5 | Alumni LPDP | Lulusan program |
| 6 | Polemik LPDP | Isu dan kontroversi |
| 7 | Penerima LPDP | Penerima dana |
| 8 | Dana LPDP | Pengelolaan keuangan |
| 9 | LPDP Luar Negeri | Studi di luar negeri |
| 10 | Seleksi LPDP | Proses pendaftaran |
| 11 | LPDP 2024 | Artikel tahun ini |
| 12 | LPDP 2023 | Artikel tahun lalu |
| 13 | LPDP S2 | Fokus magister |
| 14 | LPDP S3 | Fokus doktoral |
| 15 | Program LPDP | Program beasiswa |
| 16 | Pendaftar LPDP | Calon pendaftar |
| 17 | Kuliah LPDP | Studi dengan LPDP |
| 18 | Scholarship LPDP | Mix EN/ID terms |
| 19 | LPDP Indonesia | Konteks nasional |
| 20 | Mahasiswa LPDP | Mahasiswa penerima |

> 💡 **Strategi**: Menggunakan 20 keywords untuk menangkap berbagai aspek, timeline, dan sudut pandang dari topik LPDP
> 
> 📊 **Target**: ~5000 artikel mentah (250 × 20) → ~2500 artikel unik setelah deduplikasi

In [21]:
# Variasi kata kunci untuk memaksimalkan jumlah artikel yang ditarik
# Note: Gunakan '+' untuk mengganti spasi agar kompatibel dengan Google News RSS
kata_kunci = [
    # Set 1: Aspek Program (10 keywords)
    'LPDP',
    'Beasiswa+LPDP',
    'Awardee+LPDP',
    'Wawancara+LPDP',
    'Alumni+LPDP',
    'Polemik+LPDP',
    'Penerima+LPDP',
    'Dana+LPDP',
    'LPDP+Luar+Negeri',
    'Seleksi+LPDP',
    'LPDP+2024',
    'LPDP+2023',
    'LPDP+S2',
    'LPDP+S3',
    'Program+LPDP',
    'Pendaftar+LPDP',
    'Kuliah+LPDP',
    'Scholarship+LPDP',
    'LPDP+Indonesia',
    'Mahasiswa+LPDP'
]

print(f"📋 Total kata kunci: {len(kata_kunci)}")

📋 Total kata kunci: 20


## 🌐 Proses Data Collection

Tahapan scraping:
1. **Loop** melalui setiap kata kunci
2. **Request** artikel dari Google News
3. **Extract** data penting (judul, tanggal, deskripsi, URL, sumber)
4. **Delay** 2 detik antar request untuk menghindari rate limit

> ⚠️ **Note**: Proses ini memakan waktu **~40 detik** (20 keywords × 2 detik delay)
> 
> 📊 **Expected Output**: ~5000 artikel mentah (masing-masing keyword → ~250 artikel)

In [22]:
semua_berita = []

print("🚀 Memulai proses Data Collection...\n")

# Looping untuk setiap kata kunci
for idx, keyword in enumerate(kata_kunci, 1):
    # Display keyword dengan spasi untuk readability
    display_keyword = keyword.replace('+', ' ')
    print(f"[{idx}/{len(kata_kunci)}] 🔍 Mencari: '{display_keyword}'", end=" ")
    
    try:
        # Menarik data dari Google News
        hasil_pencarian = google_news.get_news(keyword)
        
        if hasil_pencarian:
            print(f"-> ✅ {len(hasil_pencarian)} artikel")
            for artikel in hasil_pencarian:
                # Ekstraksi komponen yang dibutuhkan
                semua_berita.append({
                    'Judul': artikel.get('title'),
                    'Tanggal_Rilis': artikel.get('published date'),
                    'Deskripsi': artikel.get('description'),
                    'URL_Artikel': artikel.get('url'),
                    'Sumber_Media': artikel.get('publisher', {}).get('title', 'Unknown')
                })
        else:
            print("-> ❌ Tidak ada hasil")
            
    except Exception as e:
        print(f"-> ⚠️ Error: {str(e)}")
    
    # Jeda untuk menghindari rate limit
    if idx < len(kata_kunci):  # Skip delay pada iterasi terakhir
        time.sleep(2)

print(f"\n🎉 Data collection selesai!")
print(f"📊 Total artikel mentah terkumpul: {len(semua_berita)}")

🚀 Memulai proses Data Collection...

[1/20] 🔍 Mencari: 'LPDP' -> ✅ 251 artikel
[2/20] 🔍 Mencari: 'Beasiswa LPDP' -> ✅ 340 artikel
[3/20] 🔍 Mencari: 'Awardee LPDP' -> ✅ 117 artikel
[4/20] 🔍 Mencari: 'Wawancara LPDP' -> ✅ 99 artikel
[5/20] 🔍 Mencari: 'Alumni LPDP' -> ✅ 243 artikel
[6/20] 🔍 Mencari: 'Polemik LPDP' -> ✅ 146 artikel
[7/20] 🔍 Mencari: 'Penerima LPDP' -> ✅ 246 artikel
[8/20] 🔍 Mencari: 'Dana LPDP' -> ✅ 239 artikel
[9/20] 🔍 Mencari: 'LPDP Luar Negeri' -> ✅ 121 artikel
[10/20] 🔍 Mencari: 'Seleksi LPDP' -> ✅ 124 artikel
[11/20] 🔍 Mencari: 'LPDP 2024' -> ✅ 146 artikel
[12/20] 🔍 Mencari: 'LPDP 2023' -> ✅ 116 artikel
[13/20] 🔍 Mencari: 'LPDP S2' -> ✅ 121 artikel
[14/20] 🔍 Mencari: 'LPDP S3' -> ✅ 116 artikel
[15/20] 🔍 Mencari: 'Program LPDP' -> ✅ 113 artikel
[16/20] 🔍 Mencari: 'Pendaftar LPDP' -> ✅ 124 artikel
[17/20] 🔍 Mencari: 'Kuliah LPDP' -> ✅ 128 artikel
[18/20] 🔍 Mencari: 'Scholarship LPDP' -> ✅ 107 artikel
[19/20] 🔍 Mencari: 'LPDP Indonesia' -> ✅ 124 artikel
[20/20] 🔍 Mencari

## 🧹 Data Cleaning & Preprocessing

Proses pembersihan data:
1. **Convert** list ke DataFrame
2. **Remove duplicates** berdasarkan judul artikel
3. **Display** statistik hasil cleaning

> 📌 **Why remove duplicates?** Kata kunci yang berbeda bisa mengembalikan artikel yang sama

In [23]:
# Konversi ke Pandas DataFrame
df_berita = pd.DataFrame(semua_berita)
print(f"📋 Total artikel mentah: {len(df_berita)}")

# Data Cleaning: Menghapus duplikasi berdasarkan judul
df_bersih = df_berita.drop_duplicates(subset=['Judul'], keep='first')
print(f"✨ Total artikel UNIK: {len(df_bersih)}")

# Hitung duplikasi
jumlah_duplikat = len(df_berita) - len(df_bersih)
print(f"🗑️ Artikel duplikat dihapus: {jumlah_duplikat}")

# Persentase keberhasilan
if len(df_berita) > 0:
    persentase_unik = (len(df_bersih) / len(df_berita)) * 100
    print(f"📈 Persentase artikel unik: {persentase_unik:.1f}%")

📋 Total artikel mentah: 3145
✨ Total artikel UNIK: 1937
🗑️ Artikel duplikat dihapus: 1208
📈 Persentase artikel unik: 61.6%


## 👁️ Preview Data

Mari lihat sample data yang berhasil dikumpulkan:

In [24]:
# Tampilkan 5 artikel pertama
if len(df_bersih) > 0:
    print("📰 Sample 5 Artikel Pertama:\n")
    print(df_bersih[['Judul', 'Tanggal_Rilis', 'Sumber_Media']].head())
    
    print("\n" + "="*80)
    print("📊 Informasi Dataset:")
    print("="*80)
    print(df_bersih.info())
    
    print("\n" + "="*80)
    print("🏢 Top 5 Sumber Media:")
    print("="*80)
    print(df_bersih['Sumber_Media'].value_counts().head(10))
else:
    print("⚠️ Tidak ada data untuk ditampilkan")

📰 Sample 5 Artikel Pertama:

                                               Judul  \
0  Video Pandangan Cinta Laura soal Siapa yang La...   
1  Cinta Laura Sentil Etika Penerima LPDP: Kalau ...   
2  Cinta Laura Tanggapi Isu LPDP: "Kalau Kita Pin...   
3  VIDEO: Serap Penerima LPDP, Fithra: Pemerintah...   
4  Akademisi UMM Kritik Tajam Prioritas LPDP pada...   

                   Tanggal_Rilis     Sumber_Media  
0  Mon, 02 Mar 2026 19:47:00 GMT          20detik  
1  Tue, 03 Mar 2026 07:13:36 GMT        Suara.com  
2  Wed, 04 Mar 2026 04:24:33 GMT      Radar Kudus  
3  Wed, 04 Mar 2026 01:00:07 GMT    CNN Indonesia  
4  Tue, 03 Mar 2026 21:56:18 GMT  beritajatim.com  

📊 Informasi Dataset:
<class 'pandas.DataFrame'>
Index: 1937 entries, 0 to 3144
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Judul          1937 non-null   str  
 1   Tanggal_Rilis  1937 non-null   str  
 2   Deskripsi      1937 non-null   str  


## 🔄 Sorting Data

Mengurutkan data berdasarkan:
1. **Sumber Media** (Alphabetically)
2. **Tanggal Rilis** (Newest First)

> 📌 **Manfaat Sorting**: Memudahkan analisis per sumber berita dan timeline publikasi artikel

In [25]:
if len(df_bersih) > 0:
    print("🔄 Mengurutkan data...\n")
    
    # Convert Tanggal_Rilis to datetime for proper sorting
    # Handle various date formats from Google News
    df_bersih['Tanggal_Parsed'] = pd.to_datetime(df_bersih['Tanggal_Rilis'], errors='coerce')
    
    # Sort by Sumber_Media (ascending) and Tanggal_Parsed (descending = newest first)
    df_sorted = df_bersih.sort_values(
        by=['Sumber_Media', 'Tanggal_Parsed'], 
        ascending=[True, False],
        na_position='last'
    ).reset_index(drop=True)
    
    print("✅ Data berhasil diurutkan!\n")
    print("="*80)
    print("📊 STATISTIK PER SUMBER MEDIA")
    print("="*80)
    
    # Count articles per media source
    media_counts = df_sorted['Sumber_Media'].value_counts()
    
    print(f"\n{'No':<4} {'Sumber Media':<30} {'Jumlah Artikel':<15} {'Persentase'}")
    print("-"*80)
    
    total_articles = len(df_sorted)
    for idx, (media, count) in enumerate(media_counts.items(), 1):
        percentage = (count / total_articles) * 100
        print(f"{idx:<4} {media:<30} {count:<15} {percentage:>5.1f}%")
    
    print("-"*80)
    print(f"{'TOTAL':<4} {'':<30} {total_articles:<15} {'100.0%':>6}")
    
    # Show date range per media
    print(f"\n{'='*80}")
    print("📅 RENTANG TANGGAL PER SUMBER MEDIA (Top 10)")
    print("="*80)
    
    top_media = media_counts.head(10).index
    print(f"\n{'Sumber Media':<30} {'Artikel Terlama':<20} {'Artikel Terbaru':<20}")
    print("-"*80)
    
    for media in top_media:
        media_data = df_sorted[df_sorted['Sumber_Media'] == media]
        valid_dates = media_data['Tanggal_Parsed'].dropna()
        
        if len(valid_dates) > 0:
            oldest = valid_dates.min().strftime('%Y-%m-%d')
            newest = valid_dates.max().strftime('%Y-%m-%d')
        else:
            oldest = "N/A"
            newest = "N/A"
        
        print(f"{media:<30} {oldest:<20} {newest:<20}")
    
    # Preview sorted data
    print(f"\n{'='*80}")
    print("👁️ PREVIEW DATA TERURUT (10 Baris Pertama)")
    print("="*80)
    print(df_sorted[['Sumber_Media', 'Tanggal_Rilis', 'Judul']].head(10))
    
    # Update df_bersih with sorted version
    df_bersih = df_sorted.copy()
    
else:
    print("⚠️ Tidak ada data untuk diurutkan")

🔄 Mengurutkan data...

✅ Data berhasil diurutkan!

📊 STATISTIK PER SUMBER MEDIA

No   Sumber Media                   Jumlah Artikel  Persentase
--------------------------------------------------------------------------------
1    detikcom                       127               6.6%
2    LPDP                           126               6.5%
3    Kompas.com                     118               6.1%
4    Medcom.id                      92                4.7%
5    SINDOnews Edukasi              49                2.5%
6    Tempo.co                       47                2.4%
7    tvOneNews                      39                2.0%
8    AsatuNews.co.id                37                1.9%
9    detikNews                      37                1.9%
10   Kumparan.com                   36                1.9%
11   Suara.com                      33                1.7%
12   CNBC Indonesia                 31                1.6%
13   Kompas.id                      29                1.5%
14   Met

## 💾 Export Data Terurut (Opsional)

Jika ingin menyimpan data yang sudah diurutkan ke file terpisah:


In [26]:
# Export data yang sudah diurutkan
if len(df_bersih) > 0:
    # Save sorted data to CSV
    sorted_filename = 'dataset_lpdp_sorted.csv'
    df_bersih.to_csv(sorted_filename, index=False)
    
    print(f"✅ Data terurut disimpan ke: {sorted_filename}")
    print(f"📊 Total rows: {len(df_bersih)}")
    print(f"📋 Columns: {', '.join(df_bersih.columns)}")
    
    # Optional: Save to Excel for easier viewing
    try:
        excel_filename = 'dataset_lpdp_sorted.xlsx'
        df_bersih.to_excel(excel_filename, index=False, sheet_name='Sorted Data')
        print(f"✅ Data terurut juga disimpan ke: {excel_filename}")
    except:
        print("⚠️ Openpyxl tidak terinstall, skip export ke Excel")
        print("   Install dengan: pip install openpyxl")
else:
    print("⚠️ Tidak ada data untuk di-export")

✅ Data terurut disimpan ke: dataset_lpdp_sorted.csv
📊 Total rows: 1937
📋 Columns: Judul, Tanggal_Rilis, Deskripsi, URL_Artikel, Sumber_Media, Tanggal_Parsed
✅ Data terurut juga disimpan ke: dataset_lpdp_sorted.xlsx


## 📂 Distribusi Data ke Anggota Kelompok

Proses pembagian data:
1. **Shuffle** data untuk distribusi acak
2. **Split** menjadi 5 bagian yang merata
3. **Export** ke CSV untuk setiap anggota

### Struktur File Output:
```
dataset_lpdp_anggota_1.csv  (~20% data)
dataset_lpdp_anggota_2.csv  (~20% data)
dataset_lpdp_anggota_3.csv  (~20% data)
dataset_lpdp_anggota_4.csv  (~20% data)
dataset_lpdp_anggota_5.csv  (~20% data)
```

> 💡 **Random shuffle** memastikan setiap anggota mendapat variasi topik yang merata

In [ ]:
if len(df_bersih) > 0:
    print("📦 Memulai proses distribusi data...\n")
    
    # Mengacak urutan data untuk distribusi merata
    df_bersih = df_bersih.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Hitung ukuran per chunk
    total_rows = len(df_bersih)
    chunk_size = total_rows // 5
    
    print(f"🔢 Total data: {total_rows} artikel")
    print(f"📦 Target per anggota: ~{chunk_size} artikel\n")
    print("="*60)
    
    # Split DataFrame menggunakan slicing manual
    for i in range(5):
        start_idx = i * chunk_size
        # Chunk terakhir mengambil sisa baris
        end_idx = start_idx + chunk_size if i < 4 else total_rows
        
        chunk = df_bersih.iloc[start_idx:end_idx]
        nama_file = f'dataset_lpdp_anggota_{i+1}.csv'
        chunk.to_csv(nama_file, index=False)
        
        print(f"✅ Anggota {i+1}: {nama_file:<30} | {len(chunk):>4} baris")
    
    print("="*60)
    print("\n🎉 SUCCESS! Pipeline Scraping Selesai!")
    print("📁 File CSV siap didistribusikan ke anggota kelompok")
    
else:
    print("❌ ERROR: Gagal mendapatkan artikel unik")
    print("💡 Tips: Coba ubah kata kunci atau cek koneksi internet")

---

## ✅ Summary

### Proses yang Telah Dilakukan:
1. ✅ Setup Google News Scraper
2. ✅ Define **20 kata kunci pencarian** (2 sets)
3. ✅ Collect ~5000 artikel dari Google News
4. ✅ Clean & remove duplicates
5. ✅ Sort by Sumber Media & Tanggal
6. ✅ Split data menjadi 5 bagian
7. ✅ Export ke CSV files

### Output Files:
- `dataset_lpdp_anggota_1.csv` - `dataset_lpdp_anggota_5.csv`
- `dataset_lpdp_sorted.csv` (opsional)

### Next Steps:
- 📊 Lakukan analisis data pada masing-masing subset
- 🧮 Kembangkan model NLP/Machine Learning
- 📈 Visualisasi insights dari data

---

**© 2026 | PBA Project - LPDP News Analysis**